---

<div style="border:1px solid #ddd; border-radius:12px; padding:20px; box-shadow:0px 4px 10px rgba(0,0,0,0.1);">

<h1 style="color:#4facfe;">📊 Modelisation prédictive : prévision du profit</h1>

<p style="font-size:15px;">
Préparation et prétraitement des données, modélisation et calibrage.
</p>

<hr>

<p>
👤 <b>Kodjo Jean DEGBEVI</b><br>
📘 <a href="../README.md">Readme</a>
</p>
<p style="font-style: italic;">Copyright (c) 2026 Kodjo Jean DEGBEVI. Distribué sous licence CC BY-NC-SA 4.0.</p>

</div>

## Méthodologie
Plan d'action pour cette modélisation :
- **Sélection des Features** : Isolement rigoureux de `Sales`, `Discount`, `Sub-Category`, `Region` et `Segment`.
- **Outliers** : Traitement si nécessaire mais avec une approche non-destructriceafin de conserver les pertes et gérant l'asymétrie extrême.
- **Modélisation** : Évaluation de deux algorithmes d'ensemble (Random Forest et XGBoost).
- **Optimisation** : Recherche d'hyperparamètres via Optuna.
- **Métriques** : RMSE, MAE, R² (remises à l'échelle spatiale via transformation inverse).

## Chargement de données et Prétraitement

### Chargement et analyse

In [ ]:
import os
import sys
from pathlib import Path
CURRENT_DIR = os.getcwd()
ROOT_DIR = Path(CURRENT_DIR).parent
sys.path.append(str(ROOT_DIR))

data_dir = os.path.join(ROOT_DIR, 'data', 'processed')
model_dir = os.path.join(ROOT_DIR, 'model')
production_assets_path = os.path.join(ROOT_DIR, 'assets', 'exports')
os.makedirs(model_dir, exist_ok=True)
os.makedirs(production_assets_path, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
import optuna
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import src.utils as utils
import warnings

warnings.filterwarnings('ignore')


In [3]:
df = pd.read_csv(os.path.join(data_dir, "superstore_processed.csv"))

features = ['Sales', 'Discount', 'Sub-Category', 'Region', 'Segment']
target = 'Profit'
df_model = df[features + [target]].copy()
df_model.head()

,Sales,Discount,Sub-Category,Region,Segment,Profit
0,261.9600,0.00,Bookcases,South,Consumer,41.9136
1,731.9400,0.00,Chairs,South,Consumer,219.5820
2,14.6200,0.00,Labels,West,Corporate,6.8714
3,957.5775,0.45,Tables,South,Consumer,-383.0310
4,22.3680,0.20,Storage,South,Consumer,2.5164


In [4]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Sales", "Discount", "Profit"))

fig.add_trace(go.Box(y=df_model['Sales'], name='Sales', marker_color='rgb(8,129,211)'), row=1, col=1)
fig.add_trace(go.Box(y=df_model['Discount'], name='Discount', marker_color='rgb(214,39,40)'), row=1, col=2)
fig.add_trace(go.Box(y=df_model['Profit'], name='Profit', marker_color='rgb(44,160,44)'), row=1, col=3)

fig.update_layout(title='Analyse des Outliers', height=500, showlegend=False)
fig.show()

### Gestion des outliers
* **Les features** : Je ne fait aucun traitement ici car les algorithmes d'ensemble que j'ai choisis gèrent bien les outliers.
* **La target** : Si je la garde telle quelle, les modèles risquent de s'affoler en cherchant à corriger au mieux les erreurs produites dans la prédictions sur ces valeurs extrèmes. Et pour ne pas aussi perdre les informations sur ces transactions à profits extreèmes, je vais plutot jouer sur l'échelle des valeurs. 
   >Mon choix porte sur la méthode du pseudo-logarithme signé.

In [5]:
X = df_model[features]
y = df_model[target]

y_log = np.sign(y) * np.log1p(np.abs(y))

### Préparation

In [6]:
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=utils.seed)

# Encodage Dummy
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)

# Alignement des colonnes
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

superstore_for_prediction = {
    "X_train_encoded": X_train_enc,
    "X_test_encoded": X_test_enc,
    "y_train_log": y_train_log,
    "y_test_log": y_test_log
}
joblib.dump(superstore_for_prediction, os.path.join(data_dir, 'superstore_for_prediction.pkl'))

print(f"Nombre de features après encodage : {X_train_enc.shape[1]}")
print(f"Taille de l'ensemble d'entraînement : {X_train_enc.shape[0]} échantillons")
print(f"Taille de l'ensemble de test : {X_test_enc.shape[0]} échantillons")

Nombre de features après encodage : 23
Taille de l'ensemble d'entraînement : 7995 échantillons
Taille de l'ensemble de test : 1999 échantillons


In [7]:
dollar_rmse_scorer = make_scorer(utils.custom_dollar_rmse_func, greater_is_better=False)
cv_strategy = RepeatedKFold(n_splits=5, n_repeats=2, random_state=utils.seed)

## Fitting

### Random Forest

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
sampler_rf = optuna.samplers.TPESampler(seed=utils.seed)
study_rf = optuna.create_study(direction='maximize', study_name='Random_Forest_Optimization', sampler=sampler_rf)
study_rf.optimize(lambda trial: utils.objective_rf(trial, X_train_enc, y_train_log, dollar_rmse_scorer, cv_strategy), n_trials=55, show_progress_bar=True)

  0%|          | 0/55 [00:00<?, ?it/s]

In [10]:
best_params_rf = study_rf.best_params
print(f"Meilleurs hyperparamètres RF : {best_params_rf}")

Meilleurs hyperparamètres RF : {'n_estimators': 942, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}


In [11]:
rf = RandomForestRegressor(**best_params_rf, random_state=42, n_jobs=-1)
rf.fit(X_train_enc, y_train_log)
rf_preds_log = rf.predict(X_test_enc)

utils.evaluate("Random Forest", y_test_log, rf_preds_log)

[Random Forest] RMSE: 80.73 $ | MAE: 19.54 $ | R²: 0.7999


In [ ]:
joblib.dump(rf, os.path.join(model_dir, 'random_forest.joblib'))

### XGBoost

In [12]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
sampler_xgb = optuna.samplers.TPESampler(seed=utils.seed)
study_xgb = optuna.create_study(direction='maximize', study_name='XGBoost_Optimization', sampler=sampler_xgb)
#study_xgb.enqueue_trial({'n_estimators': 537, 'learning_rate': 0.013615967639723439, 'max_depth': 6, 'min_child_weight': 5, 'gamma': 4.101536751132022e-06, 'subsample': 0.8101869995253598, 'colsample_bytree': 0.9578511247810497, 'reg_alpha': 0.07483743473504341, 'reg_lambda': 0.00015279522095651662})
study_xgb.optimize(lambda trial: utils.objective_xgb(trial, X_train_enc, y_train_log, dollar_rmse_scorer, cv_strategy), n_trials=160, show_progress_bar=True)

  0%|          | 0/160 [00:00<?, ?it/s]

In [14]:
best_params_xgb = study_xgb.best_params
print(f"Meilleurs hyperparamètres XGBoost : {best_params_xgb}")

Meilleurs hyperparamètres XGBoost : {'n_estimators': 537, 'learning_rate': 0.013615967639723439, 'max_depth': 6, 'min_child_weight': 5, 'gamma': 4.101536751132022e-06, 'subsample': 0.8101869995253598, 'colsample_bytree': 0.9578511247810497, 'reg_alpha': 0.07483743473504341, 'reg_lambda': 0.00015279522095651662}


In [15]:
xgb = XGBRegressor(**best_params_xgb, random_state=42, n_jobs=-1, objective='reg:squarederror')
xgb.fit(X_train_enc, y_train_log)
xgb_preds_log = xgb.predict(X_test_enc)

utils.evaluate("XGBoost", y_test_log, xgb_preds_log)

[XGBoost] RMSE: 102.64 $ | MAE: 21.23 $ | R²: 0.8168


In [ ]:
joblib.dump(xgb, os.path.join(model_dir, 'xgboost.joblib'))

## Choix et mise en production

Le modèle retenu pour la production est **Random Forest** car il minimise l'erreur métier la plus importante ici : *la prédiction du profit en dollars*. <br>Malgré un R² légèrement inférieur à XGBoost, il obtient de meilleurs résultats sur les métriques directement interprétables pour le besoin fonctionnel, avec une **RMSE plus faible** et une **MAE plus faible**. Je privilégie donc la précision monétaire et la robustesse opérationnelle plutôt qu'un gain marginal de variance expliquée.

In [24]:
cat_features = ['Sub-Category', 'Region', 'Segment']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ],
    remainder='passthrough'
 )

production_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(**best_params_rf, random_state=utils.seed, n_jobs=-1))
])

production_pipeline.fit(X_train, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [26]:
pp_log = production_pipeline.predict(X_test)

utils.evaluate("Random Forest", y_test_log, pp_log)

[Random Forest] RMSE: 81.39 $ | MAE: 19.44 $ | R²: 0.8014


In [ ]:
joblib.dump(production_pipeline, os.path.join(production_assets_path, 'profit_predictor.joblib'))